In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
%matplotlib inline

In [ ]:
CSV = "Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv"   # <- change if needed
if not os.path.exists(CSV):
    hits = glob.glob("**/*Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv", recursive=True) \
         + glob.glob("/mnt/user-data/uploads/*Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv")
    CSV = hits[0]
df = pd.read_csv(CSV)
df["addiction_level"] = df["addiction_level"].fillna("None")

order = ["None", "Mild", "Moderate", "Severe"]
X = df.drop(columns=["transaction_id", "user_id", "addicted_label", "addiction_level"])
y = df["addiction_level"].map({c: i for i, c in enumerate(order)})
yb = df["addicted_label"]

Xd = pd.get_dummies(X, drop_first=True)          
X_tr, X_te, y_tr, y_te = train_test_split(Xd, y, test_size=0.2, stratify=y, random_state=SEED)
print(Xd.shape)

(7500, 14)


In [6]:
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xd, yb, test_size=0.2, stratify=yb, random_state=SEED)
gsb = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                   {"n_estimators": [300], "max_depth": [8, None], "min_samples_leaf": [5, 10, 20]},
                   cv=cv, scoring="accuracy", n_jobs=-1).fit(Xb_tr, yb_tr)
rfb = gsb.best_estimator_
pb = rfb.predict(Xb_te)
print("Best params:", gsb.best_params_)
print(f"Binary test accuracy: {accuracy_score(yb_te, pb):.4f}  (majority baseline {yb.mean():.3f})\n")
print(classification_report(yb_te, pb, target_names=["Not addicted", "Addicted"], digits=3))

Best params: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 300}
Binary test accuracy: 0.9373  (majority baseline 0.708)

              precision    recall  f1-score   support

Not addicted      0.884     0.904     0.894       438
    Addicted      0.960     0.951     0.956      1062

    accuracy                          0.937      1500
   macro avg      0.922     0.928     0.925      1500
weighted avg      0.938     0.937     0.938      1500



In [ ]:
cat_cols = ["gender", "stress_level", "academic_work_impact"]
num_cols = [c for c in X.columns if c not in cat_cols]
pre = ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
def mlp(): return Pipeline([("prep", pre), ("mlp", MLPClassifier((64, 32), alpha=1e-3, batch_size=64,
                 early_stopping=True, max_iter=300, n_iter_no_change=15, random_state=SEED))])

Xr_tr, Xr_te = X.loc[X_tr.index], X.loc[X_te.index]
nn4 = mlp().fit(Xr_tr, y_tr)
nn2 = mlp().fit(X.loc[Xb_tr.index], yb_tr)

nn4

,4-class accuracy,Binary accuracy
Neural network (MLP),0.4887,0.9233
Random forest,0.5133,0.9373
